<small>Dataset: Student_Performance.csv</small>

# Q19. Student Performance KNN

Use nearby labeled examples to classify new records, and compare three K values.


## Question from Classroom

Student Performance Classification Using KNN 
Problem statement: Classify students into predefined academic performance categories 
using KNN. 
Tasks to be performed: 
1. Load a labeled student performance dataset. 
2. Select relevant features, such as attendance, study hours and internal marks. 
3. Define or use existing performance categories as the target. 
4. Split the dataset and scale the input features. 
5. Implement KNN and test different K values. 
6. Predict categories for unseen student records. 
7. Report accuracy, precision, recall and the confusion matrix.


## Understand before coding

Choose X/y -> stratified split -> training-only imputation and scaling -> compare K=3,5,7 -> print metrics and unseen predictions.


**Dataset / practice note:** Student category thresholds are explicitly invented for practice because the attached file has no predefined category label.


Run the code cells from top to bottom with **Shift + Enter**. If an import is missing, run the following installation line once in a new cell, then restart the kernel. The line below is a comment until you remove `#`.


**Optional package setup (run once in a new Jupyter cell if needed):**

```python
%pip install pandas numpy matplotlib scikit-learn openpyxl
```


**Code layout:** 7 listed tasks → 7 code blocks. Run them in order. Imports and helper definitions are included in the relevant task. Where the PDF repeats a task number, these blocks follow its listed order. Learned preprocessing is fitted only after splitting.


## Task 1. Load a labeled student performance dataset.

This block implements the task above. Variables from earlier blocks are reused.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv("data/Student_Performance.csv").dropna(subset=["Performance Index"])
df["Extracurricular Activities"] = df["Extracurricular Activities"].map({"Yes":1,"No":0})


## Task 2. Select relevant features, such as attendance, study hours and internal marks.

This block implements the task above. Variables from earlier blocks are reused.


In [2]:
X=df.drop(columns="Performance Index")
y=df["Performance Index"]
print("Predictors:",X.columns.tolist())


Predictors: ['Hours Studied', 'Previous Scores', 'Extracurricular Activities', 'Sleep Hours', 'Sample Question Papers Practiced']


## Task 3. Define or use existing performance categories as the target.

This block implements the task above. Variables from earlier blocks are reused.


In [3]:
# Practice categories because the supplied CSV has no official category labels.
y=pd.cut(df["Performance Index"],bins=[-np.inf,40,70,np.inf],labels=["Low","Medium","High"])
print("Practice category counts:",y.value_counts().to_dict())
# X excludes Performance Index so that it cannot leak the label.


Practice category counts: {'Medium': 4933, 'Low': 2562, 'High': 2505}


## Task 4. Split the dataset and scale the input features.

This block implements the task above. Variables from earlier blocks are reused.


In [4]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

scaler=StandardScaler()
train_scaled=scaler.fit_transform(X_train)
test_scaled=scaler.transform(X_test)
print("Training-fitted scaling defined; the model pipeline applies it inside CV too.")


Training-fitted scaling defined; the model pipeline applies it inside CV too.


## Task 5. Implement KNN and test different K values.

This block implements the task above. Variables from earlier blocks are reused.


In [5]:
models={}
results=[]
for k in [3,5,7]:
    model=make_pipeline(SimpleImputer(strategy="median"),StandardScaler(),KNeighborsClassifier(n_neighbors=k))
    cv_accuracy=cross_val_score(model,X_train,y_train,cv=3,scoring="accuracy").mean()
    model.fit(X_train,y_train)
    models[k]=model
    results.append({"K":k,"Training CV accuracy":cv_accuracy,"Test accuracy":accuracy_score(y_test,model.predict(X_test))})
table=pd.DataFrame(results)

print(table.to_string(index=False))
best_k=int(table.loc[table["Training CV accuracy"].idxmax(),"K"])
print("K selected using training CV:",best_k)
model=models[best_k]
prediction=model.predict(X_test)


 K  Training CV accuracy  Test accuracy
 3              0.902501         0.9130
 5              0.906501         0.9150
 7              0.910626         0.9115
K selected using training CV: 7


## Task 6. Predict categories for unseen student records.

This block implements the task above. Variables from earlier blocks are reused.


In [6]:
print("Categories for unseen test rows:",model.predict(X_test.head(5)))


Categories for unseen test rows: ['High' 'Medium' 'Low' 'Medium' 'Medium']


## Task 7. Report accuracy, precision, recall and the confusion matrix.

This block implements the task above. Variables from earlier blocks are reused.


In [7]:
print("Accuracy:",round(accuracy_score(y_test,prediction),4))
print("Class order:",model.classes_)
print("Confusion matrix:\n",confusion_matrix(y_test,prediction,labels=model.classes_))
print(classification_report(y_test,prediction,zero_division=0))


Accuracy: 0.9115
Class order: ['High' 'Low' 'Medium']
Confusion matrix:
 [[448   0  53]
 [  0 463  49]
 [ 35  40 912]]
              precision    recall  f1-score   support

        High       0.93      0.89      0.91       501
         Low       0.92      0.90      0.91       512
      Medium       0.90      0.92      0.91       987

    accuracy                           0.91      2000
   macro avg       0.92      0.91      0.91      2000
weighted avg       0.91      0.91      0.91      2000



## Viva preparation guide



**1. What are X and y?**

X contains the predictor columns; y contains the target we want to predict. X is a 2D table and y is a 1D series.

**2. Why split training and testing data?**

Fit using the training examples. Evaluate on separate examples to estimate how well the model generalizes.

**3. What do fit() and predict() mean?**

fit() learns parameters from training examples. predict() applies the learned model to new feature rows.

**4. Why random_state=42?**

It makes random splits or model choices reproducible. The number 42 has no special mathematical meaning.

**5. What is data leakage?**

Information unavailable for a real prediction accidentally influences training. Examples: including the answer in X or fitting a scaler on test data. Fit learned preprocessing only on training data.

**6. What is overfitting?**

The model learns training-specific details/noise and does worse on unseen data. Control complexity and use validation or cross-validation.

**7. What is a hyperparameter?**

A setting chosen outside model fitting, such as max_depth, K or the number of trees. Choose it using training validation, not by repeatedly optimizing the final test result.

**8. How do I fix FileNotFoundError or KeyError?**

Open Jupyter in the praceice folder, keep data/ beside this notebook, and print df.columns.tolist() to check exact column names. Do not change X and y row counts separately.

**9. What does the confusion matrix contain?**

Rows are actual classes and columns are predicted classes, in the printed label order. For binary labels [0,1], it is [[TN,FP],[FN,TP]].

**10. Define accuracy, precision, recall and F1.**

Accuracy=(TP+TN)/N. Precision=TP/(TP+FP). Recall=TP/(TP+FN). F1=2PR/(P+R). classification_report prints per-class values and macro/weighted averages. zero_division=0 handles an undefined score.

**11. Why can accuracy be misleading?**

If most examples belong to one class, always predicting that class may give high accuracy. Inspect minority-class recall, precision, F1 and the confusion matrix.

**12. Why stratify=y?**

It approximately preserves class proportions in training and test splits. Each class needs enough examples.

**13. How does KNN classify a record?**

Measure distance to training examples, take the K nearest, and predict by their class votes.

**14. Why scale features?**

A large numeric range could dominate distance. StandardScaler subtracts the training mean and divides by training standard deviation.

**15. Why try K=3,5,7?**

The question requests multiple K values. They illustrate the bias/variance tradeoff; three candidates do not guarantee the best possible K.

**16. Why choose K by training CV?**

Repeatedly choosing the best final test score would make the test set part of tuning. We show the requested test comparisons but select K using training cross-validation.

**17. What if K is very small or large?**

Small K can follow noise and overfit. Very large K can smooth away useful local patterns and underfit.

**18. Does an odd K prevent every tie?**

No. It helps avoid a two-class even-vote tie, but multiclass votes can still tie.

**19. Is KNN eager or lazy?**

It is called lazy learning: it retains training examples and performs much of its work during prediction.

**20. What does the pipeline do?**

Impute training medians, fit a training scaler, then fit KNN. During CV each fold fits its own preprocessing, preventing leakage.

**21. Complexity?**

Brute-force prediction needs roughly O(n*p) distance work per query for n training rows and p features, plus neighbour selection. Storage is roughly O(n*p); indexed algorithms vary.

**22. How are the student categories obtained?**

This file has a numerical Performance Index, not official categories. The practice rule is Low <=40, Medium >40 and <=70, High >70. Replace this rule with the examiner's labels.

**23. Why remove Performance Index from X?**

It directly determines our derived label. Including it would leak the answer.

**24. Why no attendance feature?**

Attendance is not in this actual file. Use the supplied attributes instead of referencing a nonexistent column.


### Task-to-code map

- **Block 1:** Load a labeled student performance dataset.
- **Block 2:** Select relevant features, such as attendance, study hours and internal marks.
- **Block 3:** Define or use existing performance categories as the target.
- **Block 4:** Split the dataset and scale the input features.
- **Block 5:** Implement KNN and test different K values.
- **Block 6:** Predict categories for unseen student records.
- **Block 7:** Report accuracy, precision, recall and the confusion matrix.
